# 第 0 阶段：云端实验室的基石

欢迎来到**大模型极客实战**！在开始解剖 Transformer 之前，我们需要先搭建好我们的“手术室”——Google Colab。

本阶段的目标是让你**完全掌控免费的云端算力**，不再因为环境配置报错、由于断线导致动辄 10GB 的模型数据丢失而抓狂。
---
### 本节核心技能栈：
1. **硬件摸底**：查看当前分配到的 GPU 算力和显存状态。
2. **魔法命令**：使用 Colab 原生命令进行环境操作。
3. **磁盘持久化**：挂载 Google Drive 并构建标准的存储目录。
4. **【SOP】大模型极速下载**：稳定、断点续传地下载开源大模型至持久化磁盘。

## 1. 硬件摸底：我在用什么显卡？

在使用大模型之前，**显存 (VRAM)** 就是我们的命脉。免费版 Colab 通常分配 NVIDIA T4 (15GB 显存) 或 L4 GPU。通过 `nvidia-smi` 命令，我们可以清晰地看到当前 GPU 的型号和内存使用情况。

In [ ]:
# 这是一个系统级命令，在前面加 '!' 可以在 Notebook 中直接执行 bash 命令
!nvidia-smi

> **💡 提示**: 注意看表里的 `Volatile Uncorr. ECC` 旁边的框，`Memory-Usage` 显示的例如 `0MiB / 15360MiB`。15360MiB 大约等于 15GB。在后续的课程中，这 15GB 将是我们微调 4B (40亿参数) 模型的最大疆域。

## 2. 魔法命令 (Magic Commands) 与环境准备
Colab 预装了大多常用的库，但我们仍然需要确保 Hugging Face 生态的核心依赖是最新的。

In [ ]:
# 使用 -q (quiet) 静默安装，-U (upgrade) 更新到最新版本
!pip install -q -U torch transformers accelerate datasets huggingface_hub

import torch
import transformers

print(f"PyTorch 版本: {torch.__version__}")
print(f"Transformers 版本: {transformers.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

## 3. 磁盘持久化：挂载 Google Drive

**⚠️ 致命警告**：Colab 实例是**临时的**！如果你把 10GB 的大模型下载到默认文件夹 `/content` 下，只要你关掉网页或者被系统判定为闲置（通常是 90 分钟），**所有数据都会被直接抹除**。

为了避免每次打开 Notebook 都得重新下载半小时数据，我们必须把 Google Drive 挂载过来，当做我们的持久化硬盘。

In [ ]:
from google.colab import drive
# 这行代码会弹出一个授权窗口，请允许 Colab 访问你的 Google Drive
drive.mount('/content/drive')

print("✅ Google Drive 挂载成功！")

接下来，我们建立一套标准的“手术室”文件目录结构：

In [ ]:
import os

# 定义我们在云盘中的工作根目录
BASE_DIR = '/content/drive/MyDrive/LLM_Neurosurgery'

# 模型权重缓存目录 (极度重要，防止重复下载)
CACHE_DIR = os.path.join(BASE_DIR, 'huggingface_cache')

# 实验数据与输出目录
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

for d in [CACHE_DIR, DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"📁 目录就绪: {d}")

## 4. 【SOP】大语言模型极速下载

由于 GitHub 国内网络或者 Hugging Face 连接不稳定，我们推荐使用官方的 `snapshot_download` 配合刚刚建好的缓存目录进行下载。
在此系列实战中，我们使用小巧但极其能打的 **Qwen3.5-4B** 作为我们解剖的对象。

> **🎯 小测验 (NotebookLM Prompt)**:
> 试着查一下 Qwen3.5-4B 的模型权重文件(通常是 .safetensors) 大概有多大？这会占用我们 T4 多少显存？

In [ ]:
from huggingface_hub import snapshot_download
import time

# 我们设定要下载的模型 ID
MODEL_ID = "Qwen/Qwen3.5-4B-Instruct"

print(f"🚀 开始同步模型 [{MODEL_ID}] 到持久化目录...")
print(f"存放路径: {CACHE_DIR}")
print("⏳ 这可能需要 2-5 分钟，喝口水耐心等待。如果之前已经下载过，瞬间就会完成！")

start_time = time.time()

# snapshot_download 会非常聪明地处理断点续传和多线程下载
model_path = snapshot_download(
    repo_id=MODEL_ID, 
    cache_dir=CACHE_DIR,
    # 排除一些我们不需要在推理时下载的巨大格式文件 (如 GGUF, 有的时候包含在官方仓库里)
    ignore_patterns=["*.gguf", "*.pt", "*.ckpt"]
)

end_time = time.time()
print(f"\n🎉 模型同步完成! 用时: {end_time - start_time:.2f} 秒")
print(f"模型实际物理路径: {model_path}")

## 5. 首次唤醒：与 Qwen3.5 的第一次对话

既然模型已经乖乖躺在我们永久挂载的硬盘里了，下面我们把它从机械的硬盘调动到最核心的 **GPU 的显存 (VRAM)** 中，并进行一次最基础的对话尝试。

这也是本门极客课程的第一个小高潮——彻底验证整个下载、挂载过程没有跑偏。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("🧠 正在将模型加载到 GPU 显存中，这大概需要几分钟。此时如果开启一个新终端运行 nvidia-smi 能够看到显存占用飙升...")

# 之前 snapshot_download 时保存的缓存目录，通过这个机制能够避开重复下载
cache_dir = "/content/drive/MyDrive/LLM_Neurosurgery/huggingface_cache"
model_id = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)

# device_map="auto" 会让 transformers 自动帮我们把模型放到可用的 GPU 上
# torch_dtype="auto" 会自动使用 float16 或 bfloat16 以节约显卡显存
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    cache_dir=cache_dir,
    torch_dtype="auto",
    device_map="auto"
)

print("✅ 模型加载成功！显存已经全部就绪！")

接下来，我们借助最新的 `apply_chat_template` 工具，给大模型输入一段简单的指令，看看它在我们的 Colab GPU 上会有怎样的本能反应。

In [ ]:
messages = [
    {"role": "system", "content": "你是一名经验丰富的大语言模型架构师，说话风格硬核幽默，擅长用大白话讲解冷核聚变般的深奥技术。"},
    {"role": "user", "content": "你好，我是一名小白，决定今天要亲手解剖你的 Transformer 架构。能给我几条忠告吗？"}
]

# 使用官方推荐的 apply_chat_template 将对话字典组装成 Qwen 能看懂的特殊 prompt 字符串
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 将文本转成数字张量丢给对应的 GPU 设备
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print("\n================ 🤖 Qwen3.5 正在 GPU 上吟唱 =================\n")

# 生成回答！限制最多吐出 512 个字
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    temperature=0.7
)

# 截断输入的部分，只保留模型全新生成的文字内容
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

---
## 第一阶段通关小结
如果你顺利执行到了这里并看到 `🎉 模型同步完成`，恭喜你，你已经建好了你的专属云端实验室！

你的模型已经永久地躺在了你的 Google Drive 里。下一次打开的时候，只要再执行一遍挂载 Drive 的代码和 `snapshot_download` 代码（基本是秒过），我们就可以开始玩弄它了。

请**保存更改 (Save a copy in GitHub)**，然后我们进入真正的修仙：第 1 阶段 - PyTorch 与 Hugging Face 基础。